In [ ]:
import logging
import pathlib

import atlite
import xarray as xr

logging.basicConfig(level=logging.INFO)

/cluster/work/projects/ec85/SARAH/2010


rclone copy -P sharepoint-group:1SHAREPOINTFORREAL/data/SARAH/SID/ --include "/SIDin2010\*" /cluster/work/projects/ec85/SARAH/SID/


To use SARAH, we have to build a new cutout.  
To find the boundaries of this cutout, we are using the exclusive economic zones, so we include offshore production as well.
They are sourced from https://www.marineregions.org/gazetteer.php?p=details&id=5686.
Their boundaries are used with a one degree buffer added and rounded up/down.

x: -2 to 38, y: 55 to 76


In [ ]:
era5 = atlite.Cutout(
    path="/cluster/work/projects/ec85/SARAH/europe_2010.nc",
    module=["era5"],
    sarah_dir="/cluster/work/projects/ec85/SARAH/2010",
    x=slice(3, 33),
    y=slice(57, 72),
    time="2010",
    chunks={"time": 100},
)

In [ ]:
era5.available_features

In [ ]:
era5.prepared_features

In [ ]:
xr.open_dataset(
    "/cluster/work/projects/ec85/SARAH/2010/SIS/SISin201001010000004231000101MA.nc"
)  # .drop_vars("record_status").to_netcdf("/cluster/work/projects/ec85/SARAH/2010/modified/01/SISin201001010000004231000101MAa.nc")

In [ ]:
abv = atlite.Cutout(
    path="/cluster/work/projects/ec85/SARAH/sarah_norway_2010_abv65.nc",
    module=["gebco", "era5"],
    # sarah_dir="/cluster/work/projects/ec85/SARAH/2010/modified/01",
    gebco_path="/cluster/work/projects/ec85/SARAH/2010/gebco/grid/gebco_2023_n76.0_s55.0_w-2.0_e38.0.nc",
    x=slice(-2, 38),
    y=slice(65, 76),
    # y=slice(55, 76),
    time="2010",
    # data=era5.data
    # chunks={"time": 100},
)

In [ ]:
abv

In [ ]:
abv.prepare()

In [ ]:
atlite.Cutout(
    path="/cluster/work/projects/ec85/SARAH/sarah_norway_2010_sub65c.nc"
).prepare()

In [ ]:
atlite.Cutout(
    path="/cluster/work/projects/ec85/SARAH/sarah_norway_2010_sub65c.nc"
).prepared_features

In [ ]:
sarah.available_features

In [ ]:
sarah.prepare()

In [ ]:
sarah

In [ ]:
sarah.prepared_features

In [ ]:
cutout = atlite.Cutout(
    path="/cluster/work/projects/ec85/SARAH/europe_2011a.nc",
    data=era5.data,
    module=["sarah", "era5"],
    sarah_dir="/cluster/work/projects/ec85/SARAH/ORD53396/",
    x=slice(-13.6913, 1.7712),
    y=slice(49.9096, 60.8479),
    time="1995-01",
    chunks={"time": 100},
)

In [ ]:
cutout.prepared_features

In [ ]:
cutout1 = atlite.Cutout(
    path="my_cutout",
    # cutout_dir="/cluster/work/projects/ec85/SARAH/ORD53396/",
    module="sarah",
    xs=slice(-5, 5),
    ys=slice(65.0, 60.0),
    years=slice(1995, 1995),
    months=slice(1, 3),
    sarah_dir="/cluster/work/projects/ec85/SARAH/ORD53396/modified/",
)
cutout1.prepare()

In [ ]:
for file in pathlib.Path("/cluster/work/projects/ec85/SARAH/ORD53396/SID/").glob(
    "SIDin*.nc"
):
    xr.open_dataset(file).drop_vars("record_status").to_netcdf(
        pathlib.Path(file).parents[1]
        / "modified"
        / pathlib.Path(file).parts[-2]
        / pathlib.Path(file).name
    )

In [ ]:
path1 = next(
    pathlib.Path("/cluster/work/projects/ec85/SARAH/2010/SID/").glob("SIDin*.nc")
)

In [ ]:
(path1.parents[1] / "modified" / path1.parts[-2]).mkdir(parents=True, exist_ok=True)

In [ ]:
def convertsarah(file):
    path1 = pathlib.Path(file)
    # print(f"Worker: {file}")
    (path1.parents[1] / "modified" / path1.parts[-2]).mkdir(parents=True, exist_ok=True)
    xr.open_dataset(file).drop_vars("record_status").to_netcdf(
        path1.parents[1] / "modified" / path1.parts[-2] / path1.name
    )

In [ ]:
import multiprocessing

if __name__ == "__main__":
    cpu_count = multiprocessing.cpu_count()

    # Create a pool of processes
    with multiprocessing.Pool(cpu_count) as pool:
        # Map pool of processes over inputs
        pool.map(
            convertsarah,
            pathlib.Path("/cluster/work/projects/ec85/SARAH/2010/SIS/").glob(
                "SISin*.nc"
            ),
        )

    print("Done")

In [ ]:
pathi = next(
    pathlib.Path("/cluster/work/projects/ec85/SARAH/2010/SIS/").glob("SISin*.nc")
)

In [ ]:
xr.open_dataset(pathi).record_status.values.unique()